# Carbon-Aware Cloud Workload Scheduling Using the DE-LU Dataset

This notebook implements a reproducible **temporal carbon-aware scheduling experiment** using the supplied `DE-LU2_generation_with_estimated_carbon_intensity.csv` dataset. It creates synthetic cloud jobs, compares an earliest-feasible carbon-unaware baseline with carbon-aware schedules, and evaluates savings across multiple days and workload profiles.

## Workflow

1. Load, inspect, and clean the 15-minute carbon-intensity series.
2. Select complete days and visualize the carbon signal.
3. Generate synthetic jobs released throughout a full 24-hour day, with deadlines that may extend into the next day.
4. Build a capacity-constrained earliest-feasible baseline and record jobs that cannot be placed before their deadlines.
5. Formulate temporal optimization as a mixed-integer linear program (MILP).
6. Compare the baseline, a simple forecast-informed policy, and a perfect-foresight oracle.
7. Evaluate emissions, energy, delay, capacity, and SLA compliance.
8. Repeat across multiple days and workload seeds; run a capacity sensitivity analysis.
9. Export schedules and experiment results to CSV.

## Interpretation and assumptions

- The grid input is the dataset column `Estimated carbon intensity (gCO2eq/kWh)`. It is an estimated lifecycle-based average intensity signal, not a direct measurement of the marginal emissions caused by an incremental task.
- The CSV has one row per production type per interval. The same interval-level carbon-intensity value is repeated across those rows; this notebook deduplicates the interval labels before scheduling.
- Each scheduling slot is 15 minutes. Job power is assumed constant while a job runs; jobs are non-preemptive and execute continuously once started.
- Each experiment uses a 72-hour context horizon: jobs are released during the first 48 hours (the preceding day and target day), while the third day provides room for target-day jobs to finish. Scheduling constraints are enforced across all 72 hours. Reported workload energy and emissions are calculated from the execution slots that fall inside the middle 24-hour target interval; jobs may begin before or finish after that interval. Deadlines are absolute slot indices from the start of the first day.
- Jobs are synthetic because the electricity dataset contains no cloud-job telemetry.
- All policies compare the same accepted jobs under the same release times, deadlines, power budget, concurrency limit, and task durations. Jobs that the earliest-feasible baseline cannot fit before their deadlines are logged and excluded consistently from that scenario's policy comparison.
- **Retrospective oracle:** uses the observed carbon-intensity trace for the preceding, target, and following day. It is an upper-bound benchmark, not a deployable policy, because it assumes perfect knowledge of the full evaluation horizon.
- **Forecast-informed policy:** uses the trace from two days before as the persistence forecast for the preceding day, the preceding day's trace for the target day, and the target day's trace for the following day.
- Dates affected by daylight-saving transitions and days with incomplete carbon data are excluded from the 72-hour experiment. The cleaning step keeps source order, which is important where local timestamps repeat or jump at daylight-saving transitions.

The model estimates workload emissions as `power (kW) × slot duration (h) × carbon intensity (gCO2eq/kWh)`, summed over active slots and converted from grams to kilograms. It does **not** use the dataset's `Estimated total emissions` as if they were cloud-job emissions.

## 0. Environment setup

Required packages: `pandas`, `numpy`, `matplotlib`, and `scipy` with `scipy.optimize.milp` support (SciPy 1.9 or later). If they are missing in your Jupyter environment, install them in a separate cell with:

```python
%pip install pandas numpy matplotlib "scipy>=1.9"
```

The notebook does not install packages automatically.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import coo_matrix

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SLOT_HOURS = 0.25
SLOTS_PER_HOUR = int(round(1 / SLOT_HOURS))
SLOTS_PER_DAY = int(round(24 / SLOT_HOURS))
INTERVAL_COL = "MTU (CET/CEST)"
CI_COL = "Estimated carbon intensity (gCO2eq/kWh)"
DATA_FILENAME = "data/DE-LU_generation_with_estimated_carbon_intensity.csv"

# Main experiment settings. Change these to explore other workload assumptions.
P_MAX_KW = 22.0
MAX_CONCURRENT_JOBS = 4
N_JOBS = 12
BASE_SEED = 42
N_DAYS_EVALUATION = 7
WORKLOAD_SEEDS = [41, 42, 43]

P_MAX_KW = 500.0
MAX_CONCURRENT_JOBS = 60
N_JOBS = 300
BASE_SEED = 42
N_DAYS_EVALUATION = 14
WORKLOAD_SEEDS = [41, 42, 43]

print("Core settings")
print(f"Slot duration: {SLOT_HOURS:.2f} hours ({SLOT_HOURS * 60:.0f} minutes)")
print(f"Power cap: {P_MAX_KW:.1f} kW | Concurrent-job cap: {MAX_CONCURRENT_JOBS}")
print(f"Synthetic workload: {N_JOBS} jobs | Workload seeds: {WORKLOAD_SEEDS}")

In [ ]:
# Prefer a CSV next to the notebook. /mnt/data is included for this provided execution environment.
path_candidates = [
    Path(DATA_FILENAME),
    Path.cwd() / DATA_FILENAME,
    Path("/mnt/data") / DATA_FILENAME,
]
DATA_PATH = next((p.resolve() for p in path_candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME}. Put the CSV in the same folder as this notebook "
        "or change DATA_PATH to the correct file location."
    )

print(f"Loading: {DATA_PATH}")
raw = pd.read_csv(DATA_PATH, low_memory=False)

required_columns = {INTERVAL_COL, CI_COL}
missing_columns = required_columns - set(raw.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print(f"Raw rows: {len(raw):,}")
print(f"Raw columns: {len(raw.columns)}")
print(f"Unique interval labels: {raw[INTERVAL_COL].nunique():,}")
print(f"Production types: {raw['Production Type'].nunique() if 'Production Type' in raw.columns else 'not available'}")
display(raw.head(3))

## 1. Clean the carbon-intensity series

The source contains rows by production type, but the scheduling model needs one carbon-intensity value per interval. The group-level `first()` below selects the first non-null carbon-intensity value in each interval. We also check whether different production-type rows disagree on the interval-level intensity.

The source interval order is preserved rather than sorting only on parsed local timestamps. This avoids accidentally reordering ambiguous local times around daylight-saving transitions.

In [ ]:
# Check whether intensity is consistent across production-type rows within each interval.
non_null_unique_ci = raw.groupby(INTERVAL_COL, sort=False)[CI_COL].nunique(dropna=True)
inconsistent_intervals = int((non_null_unique_ci > 1).sum())
print(f"Intervals with inconsistent non-null intensity values: {inconsistent_intervals}")
if inconsistent_intervals:
    warnings.warn("Some interval labels contain multiple carbon-intensity values; inspect the source before proceeding.")

# Keep the source interval order and attach one carbon-intensity value per interval.
intervals = raw[[INTERVAL_COL]].drop_duplicates(keep="first").reset_index(drop=True)
ci_by_interval = raw.groupby(INTERVAL_COL, sort=False)[CI_COL].first()
intervals[CI_COL] = intervals[INTERVAL_COL].map(ci_by_interval).astype(float)
intervals["interval_index"] = np.arange(len(intervals), dtype=int)

# Extract the local start label while ignoring a possible '(CET)'/'(CEST)' suffix.
intervals["start_text"] = intervals[INTERVAL_COL].str.extract(
    r"^(\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2})", expand=False
)
intervals["start_local"] = pd.to_datetime(
    intervals["start_text"], format="%d/%m/%Y %H:%M:%S", errors="coerce"
)
if intervals["start_local"].isna().any():
    raise ValueError("Some interval start labels could not be parsed. Inspect the MTU column format.")

intervals["date"] = intervals["start_local"].dt.strftime("%Y-%m-%d")
intervals["clock"] = intervals["start_local"].dt.strftime("%H:%M")

print(f"Unique scheduling intervals: {len(intervals):,}")
print(f"Intervals with valid intensity: {intervals[CI_COL].notna().sum():,}")
print(f"Intervals with missing intensity: {intervals[CI_COL].isna().sum():,}")
print(f"Intensity range: {intervals[CI_COL].min():.2f}–{intervals[CI_COL].max():.2f} gCO2eq/kWh")
display(intervals[["interval_index", INTERVAL_COL, "start_local", CI_COL]].head(5))

In [ ]:
# Daily coverage diagnostics. A complete 24-hour day has 96 quarter-hour slots and no missing CI values.
day_stats = (
    intervals.groupby("date", sort=False)
    .agg(
        n_slots=("interval_index", "size"),
        valid_ci_slots=(CI_COL, "count"),
        mean_ci_g_per_kwh=(CI_COL, "mean"),
        min_ci_g_per_kwh=(CI_COL, "min"),
        max_ci_g_per_kwh=(CI_COL, "max"),
    )
    .reset_index()
    .sort_values("date")
    .reset_index(drop=True)
)
day_stats["complete_96_slot_day"] = (day_stats["n_slots"] == SLOTS_PER_DAY) & (day_stats["valid_ci_slots"] == SLOTS_PER_DAY)
complete_days = day_stats.loc[day_stats["complete_96_slot_day"], "date"].tolist()
complete_day_set = set(complete_days)

print(f"Calendar dates represented: {len(day_stats):,}")
print(f"Complete 96-slot days with valid intensity: {len(complete_days):,}")
if not complete_days:
    raise ValueError("No complete 96-slot days with valid carbon intensity were found.")

# The demo needs two complete days before the target (for persistence forecasting),
# plus the target and following days for the 72-hour context.
def _previous_date(date_string: str) -> str:
    return (pd.Timestamp(date_string) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

def _next_date(date_string: str) -> str:
    return (pd.Timestamp(date_string) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

eligible_demo_dates = [
    d for d in complete_days
    if _previous_date(_previous_date(d)) in complete_day_set
    and _previous_date(d) in complete_day_set
    and _next_date(d) in complete_day_set
]
if not eligible_demo_dates:
    raise ValueError(
        "The dataset needs at least four consecutive complete days (two preceding, target, following) "
        "to run the 72-hour demonstration and persistence forecast."
    )

DEMO_DATE = eligible_demo_dates[-1]
DEMO_NEXT_DATE = _next_date(DEMO_DATE)
last_valid_row = intervals.loc[intervals[CI_COL].notna()].iloc[-1]
print(f"Last valid interval label: {last_valid_row[INTERVAL_COL]}")
print("Recent daily coverage:")
display(day_stats.tail(12))
print(f"Target demonstration date: {DEMO_DATE}; context: {_previous_date(_previous_date(DEMO_DATE))}, {_previous_date(DEMO_DATE)}, {DEMO_DATE}, {DEMO_NEXT_DATE}")
print("Dates with incomplete data and daylight-saving transition days are excluded from this experiment.")

## 2. Visualize the selected day's carbon signal

The chart uses slot position (elapsed quarter-hour index) on the x-axis. This is intentional: using the original local-clock labels directly can make daylight-saving transitions appear as duplicate or missing time intervals.

In [ ]:
def get_day_frame(date_string: str) -> pd.DataFrame:
    """Return a date's interval rows in original source order."""
    frame = intervals.loc[intervals["date"] == date_string].copy().reset_index(drop=True)
    return frame


def previous_calendar_day(date_string: str) -> str:
    return (pd.Timestamp(date_string) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def next_calendar_day(date_string: str) -> str:
    return (pd.Timestamp(date_string) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def get_complete_day_ci(date_string: str) -> np.ndarray:
    day = get_day_frame(date_string)
    values = day[CI_COL].to_numpy(dtype=float)
    if len(values) != SLOTS_PER_DAY or not np.isfinite(values).all():
        raise ValueError(f"{date_string} is not a complete 96-slot day with valid CI values.")
    return values


DEMO_PREVIOUS_DATE = previous_calendar_day(DEMO_DATE)
DEMO_NEXT_DATE = next_calendar_day(DEMO_DATE)
demo_previous_day = get_day_frame(DEMO_PREVIOUS_DATE)
demo_day = get_day_frame(DEMO_DATE)
demo_next_day = get_day_frame(DEMO_NEXT_DATE)

demo_previous_ci_day = get_complete_day_ci(DEMO_PREVIOUS_DATE)
demo_ci_day = get_complete_day_ci(DEMO_DATE)
demo_next_ci_day = get_complete_day_ci(DEMO_NEXT_DATE)

# The target day is the middle 24-hour interval: [slot 96, slot 192).
TARGET_START_SLOT = SLOTS_PER_DAY
TARGET_END_SLOT = 2 * SLOTS_PER_DAY
TARGET_WINDOW = (TARGET_START_SLOT, TARGET_END_SLOT)

# Three-day context: preceding day, target day, following day.
demo_ci = np.concatenate([demo_previous_ci_day, demo_ci_day, demo_next_ci_day])
slot_hours = np.arange(SLOTS_PER_DAY) * SLOT_HOURS
plt.figure(figsize=(14, 4.5))
plt.plot(slot_hours, demo_ci_day, linewidth=1.2)
plt.title(f"DE-LU estimated carbon intensity — target day {DEMO_DATE}")
plt.xlabel("Elapsed time from start of target day (hours)")
plt.ylabel("Carbon intensity (gCO₂eq/kWh)")
plt.xticks(np.arange(0, 24.1, 2))
plt.xlim(0, 24)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print("Target-day carbon-intensity summary:")
print(pd.Series(demo_ci_day, name="gCO2eq/kWh").describe().to_frame())
print(f"Scheduling context: {len(demo_ci)} slots ({len(demo_ci) * SLOT_HOURS:.0f} hours).")
print(f"Metrics window: slots [{TARGET_START_SLOT}, {TARGET_END_SLOT}) — {DEMO_DATE}.")


## 3. Generate a flexible synthetic workload

Each job has a release slot, duration, power demand, and deadline. Release times are distributed across the full 24-hour day. Deadlines are relative to release time but stored as absolute slot indices, so they may extend into the following day. For example, slot 92 is 23:00; a five-hour deadline window is slot 112, or 04:00 the next day. Durations are multiples of 15 minutes. Jobs are non-preemptive: once started, a job occupies its power demand for every slot until it finishes.

The workload generator is deterministic for a given random seed. It creates deadline windows of 4–12 hours (never shorter than the job duration). Resource feasibility is checked by the baseline scheduler. Jobs that cannot be placed before their deadlines are collected in a violation log and excluded from all policies in that scenario, so comparisons use a common task set.

In [ ]:
import numpy as np
import pandas as pd


def generate_workload(
    n_slots: int = 2 * SLOTS_PER_DAY,
    n_jobs: int = N_JOBS,
    seed: int = BASE_SEED,
    p_max_kw: float = P_MAX_KW,
    max_concurrent_jobs: int = MAX_CONCURRENT_JOBS,
    min_deadline_hours: float = 4.0,
    max_deadline_hours: float = 12.0,
) -> pd.DataFrame:
    """Generate jobs over the preceding and target days; deadlines may extend into day three.

    ``n_slots`` defines the release window (normally 192 slots). The 288-slot
    scheduling horizon is supplied separately through the carbon-intensity profile.
    """
    if n_slots < 1:
        raise ValueError("n_slots must be positive.")
    if n_jobs < 1:
        raise ValueError("n_jobs must be positive.")
    if p_max_kw < 3:
        raise ValueError("p_max_kw must be at least 3 kW.")
    if max_concurrent_jobs < 1:
        raise ValueError("max_concurrent_jobs must be positive.")
    if min_deadline_hours <= 0 or max_deadline_hours < min_deadline_hours:
        raise ValueError("Deadline-window bounds must satisfy 0 < min_deadline_hours <= max_deadline_hours.")

    rng = np.random.default_rng(seed)
    durations = rng.choice(
        [4, 8, 12, 16],
        size=n_jobs,
        p=[0.25, 0.35, 0.25, 0.15],
    ).astype(int)
    powers = rng.choice(
        [3, 4, 5, 6, 7, 8],
        size=n_jobs,
        p=[0.10, 0.15, 0.20, 0.25, 0.20, 0.10],
    ).astype(float)
    powers = np.minimum(powers, p_max_kw)

    # Stratified random release times spread jobs across the preceding and target days.
    # Explicitly include the first and last slots when there are at least two jobs.
    if n_jobs == 1:
        release_slots = np.array([int(rng.integers(0, n_slots))], dtype=int)
    else:
        positions = (np.arange(n_jobs) + rng.random(n_jobs)) * n_slots / n_jobs
        release_slots = np.floor(positions).astype(int)
        release_slots[0] = 0
        release_slots[-1] = n_slots - 1
        rng.shuffle(release_slots)
    release_slots = np.clip(release_slots, 0, n_slots - 1)

    min_window_slots = int(np.ceil(min_deadline_hours / SLOT_HOURS))
    max_window_slots = int(np.floor(max_deadline_hours / SLOT_HOURS))
    minimum_windows = np.maximum(durations, min_window_slots)
    if np.any(minimum_windows > max_window_slots):
        raise ValueError(
            "The maximum deadline window is shorter than at least one generated task. "
            "Increase max_deadline_hours or reduce maximum task duration."
        )
    deadline_window_slots = np.array([
        int(rng.integers(int(low), max_window_slots + 1))
        for low in minimum_windows
    ], dtype=int)
    deadline_slots = release_slots + deadline_window_slots

    tasks = pd.DataFrame({
        "task_id": [f"J{i + 1:03d}" for i in range(n_jobs)],
        "release_slot": release_slots,
        "duration_slots": durations,
        "duration_hours": durations * SLOT_HOURS,
        "power_kw": powers,
        "deadline_window_slots": deadline_window_slots,
        "deadline_after_release_hours": deadline_window_slots * SLOT_HOURS,
        "deadline_slot": deadline_slots,
    })
    return tasks.sort_values(["release_slot", "task_id"]).reset_index(drop=True)


def slot_to_clock(slot: int) -> str:
    """Format an absolute slot; label times after midnight with their day offset."""
    absolute_slot = int(slot)
    day_offset, slot_in_day = divmod(absolute_slot, SLOTS_PER_DAY)
    minutes = slot_in_day * int(round(SLOT_HOURS * 60))
    clock = f"{minutes // 60:02d}:{minutes % 60:02d}"
    return clock if day_offset == 0 else f"{clock} (+{day_offset}d)"


tasks = generate_workload(
    n_slots=2 * SLOTS_PER_DAY,
    n_jobs=N_JOBS,
    seed=BASE_SEED,
    p_max_kw=P_MAX_KW,
    max_concurrent_jobs=MAX_CONCURRENT_JOBS,
)

workload_display = tasks.copy()
workload_display["release_time"] = workload_display["release_slot"].map(slot_to_clock)
workload_display["deadline_time"] = workload_display["deadline_slot"].map(slot_to_clock)

display(workload_display[[
    "task_id", "release_time", "duration_hours", "power_kw",
    "deadline_after_release_hours", "deadline_time",
]])
print(
    f"Generated {len(tasks)} jobs across the preceding and target days, with releases from "
    f"{tasks['release_slot'].min()} to {tasks['release_slot'].max()} slots "
    f"({slot_to_clock(tasks['release_slot'].min())}–{slot_to_clock(tasks['release_slot'].max())})."
)

## 4. Carbon-unaware baseline: earliest feasible execution

The baseline is an FCFS-like policy: jobs are considered by release time and task ID, and each starts at its earliest slot that satisfies the power budget, concurrency limit, release time, and deadline. The scheduling horizon covers 48 hours, although release times are confined to the first 24 hours. If a job cannot be placed before its deadline, the scheduler records the task and failure reason instead of raising an exception. Those jobs are then excluded from the baseline, forecast, and oracle comparison for that workload scenario.

In [ ]:
def schedule_earliest_feasible(
    tasks: pd.DataFrame,
    n_slots: int,
    p_max_kw: float,
    max_concurrent_jobs: int | None,
) -> tuple[dict[str, int], pd.DataFrame]:
    """Schedule feasible jobs earliest-first and return a report for jobs that cannot be placed."""
    power_load = np.zeros(n_slots, dtype=float)
    job_load = np.zeros(n_slots, dtype=int)
    starts: dict[str, int] = {}
    violations: list[dict] = []

    ordered = tasks.sort_values(["release_slot", "task_id"])
    for row in ordered.itertuples(index=False):
        task_id = str(row.task_id)
        earliest = max(0, int(row.release_slot))
        duration = int(row.duration_slots)
        deadline = int(row.deadline_slot)
        power = float(row.power_kw)
        latest_start = min(deadline - duration, n_slots - duration)
        chosen = None
        failure_reason = None

        if power > p_max_kw + 1e-9:
            failure_reason = "task_power_exceeds_capacity"
        elif latest_start < earliest:
            failure_reason = "no_start_window_before_deadline_within_horizon"
        else:
            for start in range(earliest, latest_start + 1):
                end = start + duration
                power_ok = np.all(power_load[start:end] + power <= p_max_kw + 1e-9)
                concurrency_ok = (
                    True if max_concurrent_jobs is None
                    else np.all(job_load[start:end] + 1 <= max_concurrent_jobs)
                )
                if power_ok and concurrency_ok:
                    chosen = start
                    break
            if chosen is None:
                failure_reason = "resource_capacity_or_concurrency_conflict_before_deadline"

        if chosen is None:
            violations.append({
                "task_id": task_id,
                "release_slot": earliest,
                "duration_slots": duration,
                "power_kw": power,
                "deadline_slot": deadline,
                "failure_reason": failure_reason,
            })
            continue

        end = chosen + duration
        power_load[chosen:end] += power
        job_load[chosen:end] += 1
        starts[task_id] = int(chosen)

    columns = ["task_id", "release_slot", "duration_slots", "power_kw", "deadline_slot", "failure_reason"]
    return starts, pd.DataFrame(violations, columns=columns)


# Keep the original generated workload for reporting. All policies use only the jobs
# that the baseline can place within the deadline and resource constraints.
generated_tasks = tasks.copy()
baseline_starts, baseline_violations = schedule_earliest_feasible(
    generated_tasks, len(demo_ci), P_MAX_KW, MAX_CONCURRENT_JOBS
)
accepted_task_ids = set(baseline_starts)
tasks = generated_tasks[generated_tasks["task_id"].isin(accepted_task_ids)].copy().reset_index(drop=True)
if tasks.empty:
    raise ValueError(
        "No generated jobs could be scheduled. Increase power/concurrency capacity, "
        "widen the deadline windows, or reduce N_JOBS."
    )

demo_violation_log = baseline_violations.copy()
if not demo_violation_log.empty:
    demo_violation_log["date"] = DEMO_DATE
    demo_violation_log["workload_seed"] = BASE_SEED
    demo_violation_log["experiment"] = "demo"
else:
    demo_violation_log = pd.DataFrame(columns=list(baseline_violations.columns) + ["date", "workload_seed", "experiment"])

print(f"Generated jobs: {len(generated_tasks)}")
print(f"Accepted for policy comparison: {len(tasks)}")
print(f"Excluded after deadline/resource feasibility check: {len(baseline_violations)}")
if not baseline_violations.empty:
    print("Jobs excluded from all policies for the demonstration workload:")
    display(baseline_violations)
print(f"Baseline scheduled {len(baseline_starts)} of {len(generated_tasks)} generated jobs.")

## 5. Temporal optimization using a MILP

For each accepted job, the model creates a binary decision variable for every feasible start slot across the 72-hour horizon. Exactly one start slot must be selected per job. The objective minimizes estimated emissions using the carbon-intensity profile for the full horizon:

\[
\min E_{kg}=\frac{1}{1000}\sum_t CI_t\,\Delta t\sum_i P_i z_{i,t}
\]

where `CI` is in gCO₂eq/kWh, `P` is in kW, and `Δt = 0.25 h`. The constraints ensure that every job starts after release, finishes by its absolute deadline, and respects both the aggregate power capacity and the maximum concurrent-job count in every slot. Releases are limited to the first day, while deadlines may be on the following day.

The optimizer accepts an objective carbon-intensity profile. That lets the same model run in oracle mode (observed current-day plus following-day profiles) or forecast-informed mode (a three-day persistence forecast), while evaluation always uses the observed carbon-intensity profile for both days.

**Runtime note:** the solver uses a 5-second time limit per solve and a 1% relative MIP gap target. If it stops before proving optimality but returns an integer-feasible incumbent, the notebook uses that incumbent and validates its resource/SLA constraints. Jobs are pre-filtered using a feasible baseline schedule so an individually or cumulatively infeasible deadline does not abort the MILP comparison.

In [ ]:
MILP_SOLVE_LOG: list[dict] = []


def optimize_schedule_milp(
    tasks: pd.DataFrame,
    objective_ci: np.ndarray,
    p_max_kw: float,
    max_concurrent_jobs: int | None,
    time_limit_seconds: int = 600,
) -> dict[str, int]:
    """Minimize emissions over feasible non-preemptive task start times using SciPy/HiGHS MILP."""
    ci = np.asarray(objective_ci, dtype=float)
    n_slots = len(ci)
    job_table = tasks.reset_index(drop=True).copy()
    n_jobs = len(job_table)

    if n_slots == 0 or not np.isfinite(ci).any():
        raise ValueError("The objective carbon-intensity profile is empty or entirely missing.")

    # A binary variable represents one possible start for one job: (job_index, start, end).
    candidates: list[tuple[int, int, int]] = []
    objective: list[float] = []
    for j, row in job_table.iterrows():
        duration = int(row["duration_slots"])
        earliest = int(row["release_slot"])
        latest_start = int(row["deadline_slot"]) - duration
        feasible_count = 0
        for start in range(earliest, latest_start + 1):
            end = start + duration
            if end > n_slots:
                continue
            interval_ci = ci[start:end]
            # Never schedule a job across a missing carbon-intensity observation.
            if not np.isfinite(interval_ci).all():
                continue
            candidates.append((j, start, end))
            energy_kwh = float(row["power_kw"]) * duration * SLOT_HOURS
            emissions_kg = float(row["power_kw"]) * SLOT_HOURS * float(interval_ci.sum()) / 1000.0
            objective.append(emissions_kg)
            feasible_count += 1
        if feasible_count == 0:
            raise ValueError(
                f"No carbon-data-valid start window exists for task {row['task_id']}. "
                "Check that the task passed the baseline feasibility filter and that the "
                "carbon-intensity profile covers its complete deadline window."
            )

    n_vars = len(candidates)
    if n_vars == 0:
        raise ValueError("No feasible task start variables were generated.")

    # Constraint rows: one equality per job, one power cap per slot, optional concurrency cap per slot.
    n_power_rows = n_slots
    n_concurrency_rows = n_slots if max_concurrent_jobs is not None else 0
    n_constraints = n_jobs + n_power_rows + n_concurrency_rows
    row_indices: list[int] = []
    col_indices: list[int] = []
    values: list[float] = []

    for k, (j, start, end) in enumerate(candidates):
        row_indices.append(j)
        col_indices.append(k)
        values.append(1.0)
        power_kw = float(job_table.iloc[j]["power_kw"])
        for slot in range(start, end):
            row_indices.append(n_jobs + slot)
            col_indices.append(k)
            values.append(power_kw)
            if max_concurrent_jobs is not None:
                row_indices.append(n_jobs + n_slots + slot)
                col_indices.append(k)
                values.append(1.0)

    A = coo_matrix(
        (values, (row_indices, col_indices)),
        shape=(n_constraints, n_vars),
        dtype=float,
    ).tocsr()

    lower = np.zeros(n_constraints, dtype=float)
    upper = np.zeros(n_constraints, dtype=float)
    lower[:n_jobs] = 1.0
    upper[:n_jobs] = 1.0
    upper[n_jobs:n_jobs + n_slots] = float(p_max_kw)
    if max_concurrent_jobs is not None:
        upper[n_jobs + n_slots:] = int(max_concurrent_jobs)

    result = milp(
        c=np.asarray(objective, dtype=float),
        integrality=np.ones(n_vars, dtype=np.int8),
        bounds=Bounds(np.zeros(n_vars), np.ones(n_vars)),
        constraints=LinearConstraint(A, lower, upper),
        options={"time_limit": time_limit_seconds, "mip_rel_gap": 0.01},
    )
    if result.x is None:
        # The task subset passed to the optimizer has a feasible baseline schedule.
        # Use it as a transparent safety fallback rather than aborting the full experiment.
        fallback_starts, fallback_violations = schedule_earliest_feasible(
            job_table, n_slots, p_max_kw, max_concurrent_jobs
        )
        if fallback_violations.empty and len(fallback_starts) == n_jobs:
            message = (
                f"MILP returned no feasible incumbent within {time_limit_seconds}s "
                f"(status={result.status}: {result.message}); using the feasible earliest-baseline "
                "schedule as a fallback. This run is feasible but is not a carbon-optimized solution."
            )
            warnings.warn(message)
            MILP_SOLVE_LOG.append({
                "n_jobs": n_jobs,
                "n_horizon_slots": n_slots,
                "time_limit_seconds": time_limit_seconds,
                "status": int(result.status),
                "solver_message": str(result.message),
                "used_fallback": True,
            })
            return fallback_starts
        raise RuntimeError(
            f"MILP found no feasible incumbent ({result.message}), and the earliest-baseline "
            f"fallback could not schedule every accepted task. Unscheduled fallback jobs: "
            f"{len(fallback_violations)}. Review the feasibility filtering and horizon."
        )
    if not result.success:
        warnings.warn(
            f"MILP stopped before proving optimality (status={result.status}: {result.message}); "
            "using its feasible incumbent. Check the solver log and increase time_limit_seconds "
            "for final experiments if needed."
        )

    selected = np.flatnonzero(result.x > 0.5)
    if len(selected) != n_jobs:
        raise AssertionError(f"Expected {n_jobs} selected task starts, but found {len(selected)}.")

    starts: dict[str, int] = {}
    for k in selected:
        job_index, start, _ = candidates[int(k)]
        starts[str(job_table.iloc[job_index]["task_id"])] = int(start)

    MILP_SOLVE_LOG.append({
        "n_jobs": n_jobs,
        "n_horizon_slots": n_slots,
        "time_limit_seconds": time_limit_seconds,
        "status": int(result.status),
        "solver_message": str(result.message),
        "used_fallback": False,
        "objective_value": float(result.fun) if result.fun is not None else np.nan,
    })
    return starts


# Oracle schedule: uses the observed carbon-intensity trace for the demonstration day.
oracle_starts = optimize_schedule_milp(
    tasks, demo_ci, P_MAX_KW, MAX_CONCURRENT_JOBS
)
print(f"MILP found a feasible schedule for {len(oracle_starts)} jobs.")

## 6. Metrics and schedule validation

All schedules are validated over the complete 72-hour horizon. Energy and emissions are accumulated only over slots 96–191 (the target day); a job crossing either target-day boundary contributes only the energy and emissions incurred inside that interval.

For every policy, compute workload energy, estimated workload emissions, emissions savings relative to baseline, average and maximum start delay, deadline violations, peak power, peak concurrency, and average intensity weighted by workload energy. With fixed task powers and durations, total workload energy should be identical for all feasible policies; emissions change because the jobs are placed in different carbon-intensity intervals.

In [ ]:
def evaluate_schedule(
    tasks: pd.DataFrame,
    starts: dict[str, int],
    actual_ci: np.ndarray,
    p_max_kw: float,
    max_concurrent_jobs: int | None,
    metric_start_slot: int = SLOTS_PER_DAY,
    metric_end_slot: int = 2 * SLOTS_PER_DAY,
) -> tuple[dict, pd.DataFrame, np.ndarray, np.ndarray]:
    """Evaluate a schedule against the observed CI trace and return metrics plus slot loads."""
    ci = np.asarray(actual_ci, dtype=float)
    n_slots = len(ci)
    if not (0 <= metric_start_slot < metric_end_slot <= n_slots):
        raise ValueError("Metric window must be a valid interval within the scheduling horizon.")
    power_load = np.zeros(n_slots, dtype=float)
    job_load = np.zeros(n_slots, dtype=int)
    schedule_rows = []
    total_energy_kwh = 0.0
    total_emissions_kg = 0.0
    deadline_violations = 0
    release_violations = 0

    task_lookup = tasks.set_index("task_id")
    for task_id, start_value in starts.items():
        row = task_lookup.loc[task_id]
        start = int(start_value)
        duration = int(row["duration_slots"])
        end = start + duration
        power = float(row["power_kw"])
        if start < 0 or end > n_slots:
            raise AssertionError(f"Task {task_id} lies outside the scheduling horizon.")
        if not np.isfinite(ci[start:end]).all():
            raise AssertionError(f"Task {task_id} overlaps missing carbon-intensity data.")

        metric_start = max(start, metric_start_slot)
        metric_end = min(end, metric_end_slot)
        metric_slots = max(0, metric_end - metric_start)
        energy_kwh = power * metric_slots * SLOT_HOURS
        emissions_kg = (power * SLOT_HOURS * float(ci[metric_start:metric_end].sum()) / 1000.0) if metric_slots else 0.0
        total_energy_kwh += energy_kwh
        total_emissions_kg += emissions_kg
        power_load[start:end] += power
        job_load[start:end] += 1

        if start < int(row["release_slot"]):
            release_violations += 1
        if end > int(row["deadline_slot"]):
            deadline_violations += 1

        schedule_rows.append({
            "task_id": task_id,
            "release_slot": int(row["release_slot"]),
            "start_slot": start,
            "end_slot": end,
            "duration_slots": duration,
            "duration_hours": duration * SLOT_HOURS,
            "power_kw": power,
            "deadline_slot": int(row["deadline_slot"]),
            "waiting_hours": (start - int(row["release_slot"])) * SLOT_HOURS,
            "deadline_slack_hours": (int(row["deadline_slot"]) - end) * SLOT_HOURS,
            "energy_kwh": energy_kwh,
            "emissions_kgCO2eq": emissions_kg,
        })

    peak_power = float(power_load.max(initial=0.0))
    peak_jobs = int(job_load.max(initial=0))
    power_violation_slots = int(np.sum(power_load > p_max_kw + 1e-8))
    concurrency_violation_slots = (
        0 if max_concurrent_jobs is None
        else int(np.sum(job_load > max_concurrent_jobs))
    )
    weighted_ci = (total_emissions_kg * 1000.0 / total_energy_kwh) if total_energy_kwh else np.nan
    available_capacity_kwh = p_max_kw * (metric_end_slot - metric_start_slot) * SLOT_HOURS

    metrics = {
        "metric_window_start_slot": metric_start_slot,
        "metric_window_end_slot": metric_end_slot,
        "energy_kwh": total_energy_kwh,
        "emissions_kgCO2eq": total_emissions_kg,
        "weighted_CI_gCO2eq_per_kWh": weighted_ci,
        "mean_wait_hours": float(np.mean([r["waiting_hours"] for r in schedule_rows])) if schedule_rows else 0.0,
        "max_wait_hours": float(np.max([r["waiting_hours"] for r in schedule_rows])) if schedule_rows else 0.0,
        "deadline_violations": deadline_violations,
        "release_violations": release_violations,
        "peak_power_kw": peak_power,
        "peak_concurrent_jobs": peak_jobs,
        "power_violation_slots": power_violation_slots,
        "concurrency_violation_slots": concurrency_violation_slots,
        "capacity_energy_utilization_pct": 100.0 * total_energy_kwh / available_capacity_kwh if available_capacity_kwh else np.nan,
    }
    schedule_table = pd.DataFrame(schedule_rows).sort_values(["start_slot", "task_id"]).reset_index(drop=True)

    violations = [
        metrics["deadline_violations"], metrics["release_violations"],
        metrics["power_violation_slots"], metrics["concurrency_violation_slots"],
    ]
    if any(violations):
        raise AssertionError(f"Schedule validation failed. Violation counts: {violations}")
    return metrics, schedule_table, power_load, job_load


def add_clock_labels(schedule_table: pd.DataFrame) -> pd.DataFrame:
    out = schedule_table.copy()
    for col in ["release_slot", "start_slot", "end_slot", "deadline_slot"]:
        out[col.replace("_slot", "_time")] = out[col].map(slot_to_clock)
    cols = [
        "task_id", "release_time", "start_time", "end_time", "deadline_time",
        "duration_hours", "power_kw", "waiting_hours", "deadline_slack_hours",
        "energy_kwh", "emissions_kgCO2eq",
    ]
    return out[cols]


def load_profile(tasks: pd.DataFrame, starts: dict[str, int], n_slots: int) -> np.ndarray:
    load = np.zeros(n_slots, dtype=float)
    task_lookup = tasks.set_index("task_id")
    for task_id, start in starts.items():
        row = task_lookup.loc[task_id]
        end = int(start) + int(row["duration_slots"])
        load[int(start):end] += float(row["power_kw"])
    return load

# Validate baseline and oracle and confirm that task energy is invariant to scheduling policy.
base_metrics, base_schedule, base_power, base_jobs = evaluate_schedule(
    tasks, baseline_starts, demo_ci, P_MAX_KW, MAX_CONCURRENT_JOBS
)
oracle_metrics, oracle_schedule, oracle_power, oracle_jobs = evaluate_schedule(
    tasks, oracle_starts, demo_ci, P_MAX_KW, MAX_CONCURRENT_JOBS
)
assert np.isclose(base_metrics["energy_kwh"], oracle_metrics["energy_kwh"])
print("Baseline and oracle pass full-horizon release, deadline, power, and concurrency validation; metrics cover only the middle 24 hours.")

## 7. Add a forecast-informed policy

For a practical comparison without a separate forecast dataset, use the preceding calendar day's carbon-intensity trace as a same-slot persistence forecast for the current day. For the following day, use the current day's trace as the one-day persistence forecast. The three-day forecast is used to select task start times, but resulting emissions are evaluated against the observed carbon-intensity values for both the preceding, target, and following days.

In [ ]:
def get_three_day_ci(date_string: str) -> np.ndarray:
    """Observed CI: preceding day + target day + following day."""
    return np.concatenate([
        get_complete_day_ci(previous_calendar_day(date_string)),
        get_complete_day_ci(date_string),
        get_complete_day_ci(next_calendar_day(date_string)),
    ])


def get_three_day_forecast_ci(date_string: str) -> np.ndarray:
    """One-day persistence forecast aligned to the three-day context."""
    day_minus_two = previous_calendar_day(previous_calendar_day(date_string))
    day_minus_one = previous_calendar_day(date_string)
    return np.concatenate([
        get_complete_day_ci(day_minus_two),
        get_complete_day_ci(day_minus_one),
        get_complete_day_ci(date_string),
    ])


forecast_ci = get_three_day_forecast_ci(DEMO_DATE)
forecast_starts = optimize_schedule_milp(
    tasks, forecast_ci, P_MAX_KW, MAX_CONCURRENT_JOBS
)

policies = {
    "Carbon-unaware baseline": baseline_starts,
    "Previous-day forecast": forecast_starts,
    "Retrospective oracle": oracle_starts,
}

policy_metrics = {}
policy_schedules = {}
policy_loads = {}
for policy_name, policy_starts in policies.items():
    metrics, schedule_table, power_load, job_load = evaluate_schedule(
        tasks, policy_starts, demo_ci, P_MAX_KW, MAX_CONCURRENT_JOBS,
        metric_start_slot=TARGET_START_SLOT, metric_end_slot=TARGET_END_SLOT
    )
    policy_metrics[policy_name] = metrics
    policy_schedules[policy_name] = schedule_table
    policy_loads[policy_name] = power_load

baseline_emissions = policy_metrics["Carbon-unaware baseline"]["emissions_kgCO2eq"]
comparison_rows = []
for policy_name, metrics in policy_metrics.items():
    saving_kg = baseline_emissions - metrics["emissions_kgCO2eq"]
    saving_pct = 100.0 * saving_kg / baseline_emissions if baseline_emissions else np.nan
    comparison_rows.append({
        "policy": policy_name,
        **metrics,
        "emissions_savings_vs_baseline_kg": saving_kg,
        "emissions_savings_vs_baseline_pct": saving_pct,
    })
demo_metrics = pd.DataFrame(comparison_rows).set_index("policy")

print(
    f"Three-day context: {DEMO_PREVIOUS_DATE}, {DEMO_DATE}, {DEMO_NEXT_DATE}. "
    f"Jobs are released over the first 48 hours; all schedules are constrained over 72 hours. "
    f"Energy and emissions are reported only for {DEMO_DATE} (middle 24 hours)."
)
display(demo_metrics[[
    "energy_kwh", "emissions_kgCO2eq", "emissions_savings_vs_baseline_kg",
    "emissions_savings_vs_baseline_pct", "weighted_CI_gCO2eq_per_kWh",
    "mean_wait_hours", "max_wait_hours", "peak_power_kw", "peak_concurrent_jobs",
    "deadline_violations", "power_violation_slots", "concurrency_violation_slots",
]])

print("\nForecast-policy schedule:")
display(add_clock_labels(policy_schedules["Previous-day forecast"]))
print("\nOracle schedule:")
display(add_clock_labels(policy_schedules["Retrospective oracle"]))


## 8. Visualize schedules and aggregate power

The three panels show (1) observed and forecast carbon intensity, (2) aggregate power demand for each policy, and (3) task start times as a Gantt-style plot. The power chart makes the capacity restriction explicit; the Gantt plot helps show which jobs were shifted.

In [ ]:
def plot_policy_comparison(
    date_string: str,
    tasks: pd.DataFrame,
    actual_ci: np.ndarray,
    forecast_profile: np.ndarray,
    starts_by_policy: dict[str, dict[str, int]],
    p_max_kw: float,
) -> None:
    n_slots = len(actual_ci)
    hours = np.arange(n_slots) * SLOT_HOURS
    horizon_hours = n_slots * SLOT_HOURS
    fig, (ax_ci, ax_power, ax_gantt) = plt.subplots(
        3, 1, figsize=(15, 12), sharex=True,
        gridspec_kw={"height_ratios": [1.0, 1.0, 2.0]}
    )

    ax_ci.plot(hours, actual_ci, label="Observed CI (preceding + target + following day)", linewidth=1.4)
    ax_ci.plot(hours, forecast_profile, label="Three-day persistence forecast", linewidth=1.1, linestyle="--")
    ax_ci.set_ylabel("gCO₂eq/kWh")
    ax_ci.set_title(f"Carbon-aware scheduling comparison — {date_string} (72-hour context)")
    ax_ci.axvspan(24, 48, alpha=0.10, label="Target 24-hour metric window")
    ax_ci.axvline(24, linestyle=":", linewidth=1.0, label="Target day starts")
    ax_ci.axvline(48, linestyle=":", linewidth=1.0, label="Following day starts")
    ax_ci.legend(loc="best", ncol=2, fontsize=8)
    ax_ci.grid(True, alpha=0.25)

    for policy_name, starts in starts_by_policy.items():
        power = load_profile(tasks, starts, n_slots)
        ax_power.step(hours, power, where="post", label=policy_name, linewidth=1.3)
    ax_power.axhline(p_max_kw, linestyle="--", linewidth=1.2, label=f"Power capacity ({p_max_kw:g} kW)")
    ax_power.axvspan(24, 48, alpha=0.10)
    ax_power.axvline(24, linestyle=":", linewidth=1.0)
    ax_power.axvline(48, linestyle=":", linewidth=1.0)
    ax_power.set_ylabel("Aggregate power (kW)")
    ax_power.legend(loc="upper right", ncol=2, fontsize=8)
    ax_power.grid(True, alpha=0.25)

    base_y = np.arange(len(tasks)) * 4.0
    policy_offsets = {
        "Carbon-unaware baseline": 0.0,
        "Previous-day forecast": 1.0,
        "Retrospective oracle": 2.0,
    }
    for policy_name, starts in starts_by_policy.items():
        left = []
        widths = []
        y_values = []
        for j, task in tasks.reset_index(drop=True).iterrows():
            start = int(starts[str(task["task_id"])])
            duration_h = int(task["duration_slots"]) * SLOT_HOURS
            left.append(start * SLOT_HOURS)
            widths.append(duration_h)
            y_values.append(base_y[j] + policy_offsets[policy_name])
        ax_gantt.barh(y_values, widths, left=left, height=0.65, label=policy_name)

    ax_gantt.set_yticks(base_y + 1.0)
    ax_gantt.set_yticklabels(tasks["task_id"].tolist())
    ax_gantt.invert_yaxis()
    ax_gantt.set_ylabel("Job")
    ax_gantt.set_xlabel("Elapsed time from start of day (hours)")
    ax_gantt.set_xticks(np.arange(0, horizon_hours + 0.1, 2))
    ax_gantt.set_xlim(0, horizon_hours)
    ax_gantt.axvspan(24, 48, alpha=0.10)
    ax_gantt.axvline(24, linestyle=":", linewidth=1.0)
    ax_gantt.axvline(48, linestyle=":", linewidth=1.0)
    ax_gantt.grid(True, axis="x", alpha=0.25)
    ax_gantt.legend(loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.show()


plot_policy_comparison(DEMO_DATE, tasks, demo_ci, forecast_ci, policies, P_MAX_KW)

## 9. Repeat over multiple days and workload profiles

The main multi-day experiment uses complete days with complete previous- and following-day carbon-intensity profiles. Three workload seeds are evaluated on every date. Jobs are released during the current date and may complete on the following date. For each date/seed pair, the earliest-feasible baseline determines a common feasible job set; any excluded jobs and reasons are logged, and the baseline, forecast-informed policy, and oracle all operate on the same accepted jobs.

The forecast for the evaluation day is the previous calendar day's same-slot trace. The following-day forecast uses the evaluation day's trace as a one-day persistence forecast. Dates with incomplete profiles or daylight-saving transitions are excluded from this experiment.

In [ ]:
eligible_date_set = set(complete_days)
forecast_eligible_dates = [
    d for d in complete_days
    if previous_calendar_day(previous_calendar_day(d)) in eligible_date_set
    and previous_calendar_day(d) in eligible_date_set
    and next_calendar_day(d) in eligible_date_set
]
selected_dates = forecast_eligible_dates[-N_DAYS_EVALUATION:]
if not selected_dates:
    raise ValueError(
        "No evaluation dates have complete two-preceding/target/following-day CI profiles."
    )

print(f"Evaluation dates ({len(selected_dates)}): {selected_dates[0]} through {selected_dates[-1]}")
experiment_rows = []
violation_records = demo_violation_log.to_dict("records")

for date_string in selected_dates:
    actual_ci = get_three_day_ci(date_string)
    forecast_ci_day = get_three_day_forecast_ci(date_string)
    for seed in WORKLOAD_SEEDS:
        scenario_generated_tasks = generate_workload(
            n_slots=2 * SLOTS_PER_DAY, n_jobs=N_JOBS, seed=int(seed),
            p_max_kw=P_MAX_KW, max_concurrent_jobs=MAX_CONCURRENT_JOBS,
        )
        scenario_baseline_all, scenario_violations = schedule_earliest_feasible(
            scenario_generated_tasks, len(actual_ci), P_MAX_KW, MAX_CONCURRENT_JOBS
        )
        accepted_ids = set(scenario_baseline_all)
        scenario_tasks = scenario_generated_tasks[
            scenario_generated_tasks["task_id"].isin(accepted_ids)
        ].copy().reset_index(drop=True)

        for violation in scenario_violations.to_dict("records"):
            violation_records.append({
                **violation,
                "date": date_string,
                "workload_seed": int(seed),
                "experiment": "multi_day",
            })

        if scenario_tasks.empty:
            warnings.warn(
                f"Skipping {date_string}, seed {seed}: no jobs fit within the current deadline/resource constraints. "
                "The rejected jobs are recorded in deadline_violation_log."
            )
            continue

        scenario_baseline = scenario_baseline_all
        scenario_forecast = optimize_schedule_milp(
            scenario_tasks, forecast_ci_day, P_MAX_KW, MAX_CONCURRENT_JOBS
        )
        scenario_oracle = optimize_schedule_milp(
            scenario_tasks, actual_ci, P_MAX_KW, MAX_CONCURRENT_JOBS
        )
        scenario_policies = {
            "Carbon-unaware baseline": scenario_baseline,
            "Previous-day forecast": scenario_forecast,
            "Retrospective oracle": scenario_oracle,
        }
        baseline_emissions_scenario = None
        scenario_metrics = {}
        for policy_name, start_map in scenario_policies.items():
            metrics, _, _, _ = evaluate_schedule(
                scenario_tasks, start_map, actual_ci, P_MAX_KW, MAX_CONCURRENT_JOBS,
                metric_start_slot=SLOTS_PER_DAY, metric_end_slot=2 * SLOTS_PER_DAY
            )
            scenario_metrics[policy_name] = metrics
            if policy_name == "Carbon-unaware baseline":
                baseline_emissions_scenario = metrics["emissions_kgCO2eq"]

        for policy_name, metrics in scenario_metrics.items():
            base_emissions = float(baseline_emissions_scenario)
            saving_kg = base_emissions - metrics["emissions_kgCO2eq"]
            saving_pct = 100.0 * saving_kg / base_emissions if base_emissions else np.nan
            experiment_rows.append({
                "date": date_string,
                "workload_seed": int(seed),
                "n_jobs": N_JOBS,
                "generated_jobs": len(scenario_generated_tasks),
                "scheduled_jobs": len(scenario_tasks),
                "excluded_jobs": len(scenario_violations),
                "p_max_kw": P_MAX_KW,
                "max_concurrent_jobs": MAX_CONCURRENT_JOBS,
                "policy": policy_name,
                **metrics,
                "emissions_savings_vs_baseline_kg": saving_kg,
                "emissions_savings_vs_baseline_pct": saving_pct,
            })

experiment_results = pd.DataFrame(experiment_rows)
deadline_violation_log = pd.DataFrame(violation_records)
if experiment_results.empty:
    raise ValueError(
        "No multi-day scenarios had any schedulable jobs. Increase resource capacity or relax deadline windows."
    )

completed_scenarios = experiment_results[["date", "workload_seed"]].drop_duplicates().shape[0]
print(f"Completed {completed_scenarios} date/workload scenarios.")
print("Aggregate results by policy:")
experiment_summary = (
    experiment_results.groupby("policy")
    .agg(
        scenarios=("date", "count"),
        mean_generated_jobs=("generated_jobs", "mean"),
        mean_scheduled_jobs=("scheduled_jobs", "mean"),
        mean_excluded_jobs=("excluded_jobs", "mean"),
        mean_emissions_kgCO2eq=("emissions_kgCO2eq", "mean"),
        median_emissions_kgCO2eq=("emissions_kgCO2eq", "median"),
        mean_savings_kg=("emissions_savings_vs_baseline_kg", "mean"),
        mean_savings_pct=("emissions_savings_vs_baseline_pct", "mean"),
        std_savings_pct=("emissions_savings_vs_baseline_pct", "std"),
        mean_wait_hours=("mean_wait_hours", "mean"),
        mean_peak_power_kw=("peak_power_kw", "mean"),
        deadline_violations=("deadline_violations", "sum"),
    )
    .sort_index()
)
display(experiment_summary)

print(f"Jobs excluded across demo and multi-day scenarios: {len(deadline_violation_log)}")
if not deadline_violation_log.empty:
    display(deadline_violation_log)

daily_savings = experiment_results.groupby(["date", "policy"])["emissions_savings_vs_baseline_pct"].mean().unstack("policy")
plt.figure(figsize=(13, 5))
for policy_name in ["Previous-day forecast", "Retrospective oracle"]:
    if policy_name in daily_savings.columns:
        plt.plot(daily_savings.index, daily_savings[policy_name], marker="o", label=policy_name)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Mean emissions savings relative to baseline by day")
plt.xlabel("Evaluation date")
plt.ylabel("Emissions savings (%)")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
box_values = [
    experiment_results.loc[experiment_results["policy"] == name, "emissions_savings_vs_baseline_pct"].to_numpy()
    for name in ["Previous-day forecast", "Retrospective oracle"]
]
plt.boxplot(box_values, tick_labels=["Previous-day forecast", "Retrospective oracle"])
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Distribution of emissions savings across days and workloads")
plt.ylabel("Emissions savings relative to baseline (%)")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 10. Sensitivity analysis: change the power-capacity limit

Capacity is an important constraint: a tight limit can prevent jobs from occupying the lowest-carbon intervals simultaneously. For each capacity, the baseline checks feasibility against the full generated workload. Jobs it cannot schedule by their deadlines are logged and excluded from all three policies for that capacity, so each capacity comparison uses a common accepted task set.

In [ ]:
capacity_values = [18.0, 22.0, 26.0]
sensitivity_rows = []
sensitivity_violation_records = []

for cap in capacity_values:
    cap_baseline_all, cap_violations = schedule_earliest_feasible(
        generated_tasks, len(demo_ci), cap, MAX_CONCURRENT_JOBS
    )
    cap_accepted_ids = set(cap_baseline_all)
    cap_tasks = generated_tasks[generated_tasks["task_id"].isin(cap_accepted_ids)].copy().reset_index(drop=True)
    for violation in cap_violations.to_dict("records"):
        sensitivity_violation_records.append({
            **violation,
            "date": DEMO_DATE,
            "workload_seed": BASE_SEED,
            "experiment": "capacity_sensitivity",
            "p_max_kw": cap,
        })
    if cap_tasks.empty:
        warnings.warn(f"Skipping capacity {cap:g} kW because no generated jobs were schedulable.")
        continue

    cap_baseline = cap_baseline_all
    cap_forecast = optimize_schedule_milp(cap_tasks, forecast_ci, cap, MAX_CONCURRENT_JOBS)
    cap_oracle = optimize_schedule_milp(cap_tasks, demo_ci, cap, MAX_CONCURRENT_JOBS)
    cap_policies = {
        "Carbon-unaware baseline": cap_baseline,
        "Previous-day forecast": cap_forecast,
        "Retrospective oracle": cap_oracle,
    }
    cap_metrics = {}
    for name, start_map in cap_policies.items():
        metrics, _, _, _ = evaluate_schedule(
            cap_tasks, start_map, demo_ci, cap, MAX_CONCURRENT_JOBS,
            metric_start_slot=TARGET_START_SLOT, metric_end_slot=TARGET_END_SLOT
        )
        cap_metrics[name] = metrics
    cap_base_emissions = cap_metrics["Carbon-unaware baseline"]["emissions_kgCO2eq"]
    for name, metrics in cap_metrics.items():
        saving_kg = cap_base_emissions - metrics["emissions_kgCO2eq"]
        sensitivity_rows.append({
            "p_max_kw": cap,
            "policy": name,
            "generated_jobs": len(generated_tasks),
            "scheduled_jobs": len(cap_tasks),
            "excluded_jobs": len(cap_violations),
            "energy_kwh": metrics["energy_kwh"],
            "emissions_kgCO2eq": metrics["emissions_kgCO2eq"],
            "savings_vs_baseline_kg": saving_kg,
            "savings_vs_baseline_pct": 100.0 * saving_kg / cap_base_emissions if cap_base_emissions else np.nan,
            "mean_wait_hours": metrics["mean_wait_hours"],
            "peak_power_kw": metrics["peak_power_kw"],
            "deadline_violations": metrics["deadline_violations"],
        })

if sensitivity_violation_records:
    deadline_violation_log = pd.concat(
        [deadline_violation_log, pd.DataFrame(sensitivity_violation_records)],
        ignore_index=True,
        sort=False,
    )
    print("Jobs excluded during capacity sensitivity analysis:")
    display(pd.DataFrame(sensitivity_violation_records))

capacity_results = pd.DataFrame(sensitivity_rows)
if capacity_results.empty:
    raise ValueError("Capacity sensitivity produced no feasible workload scenarios.")
display(capacity_results.sort_values(["p_max_kw", "policy"]).reset_index(drop=True))

plt.figure(figsize=(9, 5))
for policy_name in ["Previous-day forecast", "Retrospective oracle"]:
    subset = capacity_results[capacity_results["policy"] == policy_name].sort_values("p_max_kw")
    plt.plot(subset["p_max_kw"], subset["savings_vs_baseline_pct"], marker="o", label=policy_name)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Carbon savings sensitivity to aggregate power capacity")
plt.xlabel("Power capacity (kW)")
plt.ylabel("Emissions savings relative to same-capacity baseline (%)")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# Inspect solver statuses and any cases where the optimizer had to fall back to the baseline.
if MILP_SOLVE_LOG:
    milp_solve_log_df = pd.DataFrame(MILP_SOLVE_LOG)
    print(
        f"MILP solves recorded: {len(milp_solve_log_df)}; "
        f"baseline fallbacks: {int(milp_solve_log_df.get('used_fallback', pd.Series(dtype=bool)).fillna(False).sum())}."
    )
    display(milp_solve_log_df)


## 11. Export results and produce a concise interpretation

The following cell writes portable CSV outputs to `carbon_scheduling_outputs/` in the notebook's current working directory. These files make it easier to inspect results outside Jupyter or reuse them in a report.

In [ ]:
#OUTPUT_DIR = Path("carbon_scheduling_outputs")
#OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
## The synthetic workload and the three schedules for the demo day.
#tasks.to_csv(OUTPUT_DIR / "demo_workload.csv", index=False)
#demo_metrics.reset_index().to_csv(OUTPUT_DIR / "demo_policy_metrics.csv", index=False)
#
#schedule_export_frames = []
#for policy_name, schedule_table in policy_schedules.items():
#    schedule_export = add_clock_labels(schedule_table)
#    schedule_export.insert(0, "policy", policy_name)
#    schedule_export.insert(1, "date", DEMO_DATE)
#    schedule_export_frames.append(schedule_export)
#pd.concat(schedule_export_frames, ignore_index=True).to_csv(
#    OUTPUT_DIR / "demo_schedules.csv", index=False
#)
#
#experiment_results.to_csv(OUTPUT_DIR / "multi_day_experiment_results.csv", index=False)
#deadline_violation_log.to_csv(OUTPUT_DIR / "deadline_violation_log.csv", index=False)
#pd.DataFrame(MILP_SOLVE_LOG).to_csv(OUTPUT_DIR / "milp_solver_log.csv", index=False)
#experiment_summary.reset_index().to_csv(OUTPUT_DIR / "multi_day_experiment_summary.csv", index=False)
#capacity_results.to_csv(OUTPUT_DIR / "capacity_sensitivity_results.csv", index=False)
#day_stats.to_csv(OUTPUT_DIR / "daily_carbon_coverage.csv", index=False)
#
#print(f"CSV outputs saved to: {OUTPUT_DIR.resolve()}")
#for file_path in sorted(OUTPUT_DIR.glob("*.csv")):
#    print(f"- {file_path.name} ({file_path.stat().st_size:,} bytes)")
#
## Core sanity checks for the detailed comparison.
#assert len(baseline_starts) == len(tasks) == len(forecast_starts) == len(oracle_starts)
#assert np.isclose(
#    policy_metrics["Carbon-unaware baseline"]["energy_kwh"],
#    policy_metrics["Previous-day forecast"]["energy_kwh"],
#)
#assert np.isclose(
#    policy_metrics["Carbon-unaware baseline"]["energy_kwh"],
#    policy_metrics["Retrospective oracle"]["energy_kwh"],
#)
#assert experiment_results["deadline_violations"].sum() == 0
#print("All final sanity checks passed.")

## 12. How to interpret the outcomes

Use the tables and plots to answer the following research questions:

1. **Does temporal shifting reduce estimated emissions?** Compare each policy's emissions and percentage savings against the same-capacity baseline.
2. **How much does forecast error matter?** Compare the previous-day forecast policy against the retrospective oracle. The gap indicates the value of better carbon-intensity forecasts for these workloads, under this particular forecast proxy.
3. **What is the capacity trade-off?** Observe how power limits influence achievable savings, peak load, and average waiting time.
4. **Are SLA constraints respected?** Deadline, release-time, power-capacity, and concurrency violations should all remain zero for accepted schedules. Report excluded jobs separately from violations observed in accepted schedules; inspect `deadline_violation_log` for each excluded task and its failure reason.
5. **Is the effect robust across days and workloads?** Use the multi-day distribution, not a single day's best result, to report means, medians, and variability.

### Limitations to state in a thesis

- Synthetic jobs do not represent measured cloud workloads; vary job counts, durations, power demand, and deadlines in further experiments.
- Jobs excluded by the earliest-feasible baseline are not included in that scenario's policy emissions comparison. Report both the accepted workload size and excluded-job count to make this filtering explicit.
- The data signal is an estimated average lifecycle carbon intensity. It is not a direct estimate of marginal emissions avoided.
- The oracle uses perfect future knowledge and is only an offline upper-bound benchmark.
- The previous-day trace is a deliberately simple forecast proxy. A stronger study should train and evaluate a forecasting model with a chronological train/test split, then feed genuinely out-of-sample forecasts to the scheduler.
- The MILP uses a 5-second time limit and a 1% relative gap target to keep multi-day experiments practical. If no feasible incumbent is returned, the notebook falls back to the earliest-feasible schedule and records this in `MILP_SOLVE_LOG`; such a fallback is feasible but is not a carbon-optimized result. Inspect solver warnings and `MILP_SOLVE_LOG`, and increase the time limit for final experiments where needed.
- This version models one region and temporal shifting only. Spatial scheduling would require equivalent traces for other regions plus network-transfer energy, latency, regional capacity, and migration constraints.
- The cloud power profile is simplified: tasks have constant power, and idle/server overhead, DVFS, memory constraints, cooling, startup costs, and task migration are not modeled.

**Recommended reporting practice:** report the dataset coverage dates, workload-generation parameters, capacity assumptions, optimization solver and status, mean/median savings, variability across scenarios, and all SLA/resource violations. Do not report the oracle result as a deployable forecast-based saving.